In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, sampler
import torchvision.datasets as dset
from torchvision import transforms as T

%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0)  # Set default size of plots.
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

device = 'cuda' if th.cuda.is_available() else 'mps' if th.backends.mps.is_available() else 'cpu'

print('using device:', device)


In [ ]:
N_TRAIN = 64 * 844
batch_size = 64

transform = T.Compose([T.ToTensor(),
                       T.Lambda(lambda t: th.flatten(t))]) # TODO: add preprocessing

mnist_train = dset.MNIST('../data', train=True, download=True, 
                         transform=transform)
loader_train = DataLoader(mnist_train, batch_size=batch_size, 
                          sampler=sampler.SubsetRandomSampler(range(N_TRAIN)))

mnist_val = dset.MNIST(root='../data', train=True, download=True, 
                       transform=transform)
loader_val = DataLoader(mnist_val, batch_size=batch_size, 
                        sampler=sampler.SubsetRandomSampler(range(N_TRAIN, len(mnist_val))))

mnist_test = dset.MNIST('../data', train=False, download=True,
                        transform=transform)
loader_test = DataLoader(mnist_test, batch_size=batch_size, 
                         shuffle=False)

_x, _y = next(iter(loader_train))
print('batch data shape:', _x.shape)
print('batch label shape:', _y.shape)
print('number of batches in train loader:', len(loader_train))
print('number of batches in val loader:', len(loader_val))
print('number of batches in test loader:', len(loader_test))
print()

print('number train samples:', len(mnist_train))
print('number val samples:', len(mnist_val) - N_TRAIN)
print('number test samples:', len(mnist_test))
print()
print('training data shape:', mnist_train.data.shape)
print('training labels shape:', mnist_train.targets.shape)
print('validation data shape:', mnist_val.data.shape)
print('validation labels shape:', mnist_val.targets.shape)
print('test data shape:', mnist_test.data.shape)
print('test labels shape:', mnist_test.targets.shape)

In [ ]:
classes = ['zero', 'one', 'two', 'three', 'four',
           'five', 'six', 'seven', 'eight', 'nine']
n_classes = len(classes)
samples_per_class = 5
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(mnist_train.targets == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * n_classes + y + 1
        plt.subplot(samples_per_class, n_classes, plt_idx)
        plt.imshow(mnist_train.data[idx])
        plt.axis('off')
        if i == 0:
            plt.title(cls)

In [ ]:
from mmidas.augmentation.networks import *
from mmidas.augmentation.train import train_augmenter

input_dim = 28 * 28
latent_dim = 64
noise_dim = 32
lr = 1e-3
n_epochs = 100
mode = 'MSE'
alpha = 1.0
lam = [1.0, 1.0, 1.0, 1.0]
saving_path = '.'
initial_w = False
save = False

parameters = {
    'initial_w': initial_w,
    'learning_rate': lr,
    'batch_size': batch_size,
    'num_epochs': n_epochs,
    'n_features': input_dim,
    'alpha': alpha,
    'lambda': lam,
    'mode': mode,
    'saving_path': saving_path,
    'save': save
}

aug = Augmenter(input_dim=input_dim, latent_dim=latent_dim, noise_dim=noise_dim).to(device)
disc = Discriminator(input_dim=input_dim).to(device)

loss_hist = train_augmenter(aug, disc, loader_train, parameters, device=device, print_every=211)

In [ ]:
plt.plot(loss_hist['aug_loss'], label='Augmenter loss')
# plt.plot(loss_hist['disc_loss'], label='Discriminator loss')
plt.xlabel('Iteration number')
plt.ylabel('Loss value')
plt.title('Augmenter Training loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.plot(loss_hist['disc_loss'], label='Discriminator loss')
plt.xlabel('Iteration number')
plt.ylabel('Loss value')
plt.title('Discriminator Training loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Now we can use the trained augmenter to augment the training data
aug.eval()
def augment_data(data, augmenter, device):
    data = data.to(device)
    with th.no_grad():
        augmented_data = augmenter(data, noise=None, device=device)
    return augmented_data

augmented_train_data = []
for x, y in loader_train:
    _, augmented_x = augment_data(x, aug, device)
    augmented_train_data.append((augmented_x, y))
augmented_train_data = th.cat([x for x, y in augmented_train_data], dim=0)

# Visualize some augmented samples
n_samples = 10
plt.figure(figsize=(12, 5))
for i in range(n_samples):
    plt.subplot(2, n_samples, i + 1)
    plt.imshow(augmented_train_data[i].view(28, 28).cpu(), cmap='gray')
    plt.axis('off')
    plt.title('Augmented')
    plt.subplot(2, n_samples, i + 1 + n_samples)
    plt.imshow(mnist_train.data[i].view(28, 28).cpu(), cmap='gray')
    plt.axis('off')
    plt.title('Original')
plt.tight_layout()
plt.show()
# Save the augmenter model